[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CS7150/classdemos/blob/main/initialization/depth-init.ipynb)

# Depth, activation, init: conditioning from the other side

A signal (one column per layer, a histogram of that layer's activations) spreads through a deep
network. Each layer computes a pre-activation $h = W \cdot a$ from the previous layer's activation
$a$, then applies a nonlinearity $a' = \phi(h)$. Whether that signal survives 40 layers, vanishes to
zero, or blows up to infinity turns out to be governed by one number: the variance of $W$.

**Why variance is the right thing to track.** Treat each entry of $a$ as a random variable with
mean zero and variance $\mathrm{Var}(a)$, and each entry of $W$ as drawn independently from a
distribution with mean zero and variance $\mathrm{Var}(W)$, independent of $a$. Each output unit of
$h$ is a sum of $n_{in}$ such products, $h_j = \sum_{i=1}^{n_{in}} W_{ji} a_i$. Since the terms are
independent and zero-mean, variances add:

$$\mathrm{Var}(h) = n_{in} \, \mathrm{Var}(W) \, \mathrm{Var}(a)$$

So every layer multiplies the signal's variance by the factor $n_{in}\mathrm{Var}(W)$ (before the
nonlinearity reshapes it further). If that factor is above 1, variance compounds geometrically over
depth and explodes; if it's below 1, it compounds down to zero. Keeping it near 1 is exactly what
the two classic initialization schemes are designed to do:

$$\text{Xavier/Glorot: } \mathrm{Var}(W) = \frac{1}{n_{in}} \qquad
  \text{He: } \mathrm{Var}(W) = \frac{2}{n_{in}}$$

Xavier init makes $n_{in}\mathrm{Var}(W) = 1$ exactly, so $\mathrm{Var}(h) = \mathrm{Var}(a)$ --
the pre-activation variance is preserved layer to layer, assuming the nonlinearity itself doesn't
change the variance much. That assumption holds reasonably well for `tanh` (which is
close to linear near 0, where most of a unit-variance signal lives) but breaks for `relu`.

**Why ReLU needs the extra factor of 2.** `relu(h) = max(0, h)` zeroes out every negative input.
For $h$ symmetric around zero, that means *half* of the units are simply set to 0 on every forward
pass, and the other half pass through unchanged. Averaged over many units, this cuts the variance of
the activation roughly in half relative to its pre-activation: $\mathrm{Var}(a') \approx
\tfrac{1}{2}\mathrm{Var}(h)$. Left uncorrected, that factor of $\tfrac12$ compounds every layer and
the signal vanishes exponentially with depth. He initialization simply doubles $\mathrm{Var}(W)$ to
cancel it out, so that the *post-activation* variance, not just the pre-activation variance, is what
gets preserved layer to layer.

The cells below build exactly this forward pass in plain NumPy -- no framework, no autograd, just
the matrix multiply and the nonlinearity -- and plot the resulting standard deviation of the signal
against depth for a few activation/init combinations, so you can see the theory above play out
numerically.

In [ ]:
#@title Setup: network + variance-tracking helper (double-click to inspect) { display-mode: "form" }
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

N = 48  # layer width, matches the interactive demo

def activate(x, kind):
    if kind == 'sigmoid':
        return 1 / (1 + np.exp(-x))
    if kind == 'tanh':
        return np.tanh(x)
    return np.maximum(0, x)  # relu

def weight_std(kind, n_in):
    if kind == 'naive':
        return 1.0
    if kind == 'small':
        return 0.1
    if kind == 'xavier':
        return np.sqrt(1 / n_in)
    return np.sqrt(2 / n_in)  # he

def plot_variance_curves(curves, colors, labels):
    plt.figure(figsize=(7, 4))
    for stds, color, label in zip(curves, colors, labels):
        plt.plot(stds, color=color, marker='o', markersize=3, label=label)
    plt.yscale('log')
    plt.xlabel('layer')
    plt.ylabel('std of activations (log scale)')
    plt.axhspan(1e-2, 1e1, color='0.9', zorder=0, label='_healthy range')
    plt.legend()
    plt.title('Signal strength vs. depth')
    plt.show()

## The forward pass

$$h = W \cdot a \qquad a = \phi(h)$$

with weight variance set by the init scheme:

$$\text{Xavier: } \mathrm{Var}(W) = \tfrac{1}{n_{in}} \qquad \text{He: } \mathrm{Var}(W) = \tfrac{2}{n_{in}}$$

Every layer is the same width `N`, so `n_in = N` throughout. This is the entire algorithm --
everything else in this notebook is just running it under different settings and plotting the
result.

In [ ]:
def forward_pass(depth, act_kind, init_kind):
    a = np.random.randn(N)  # input ~ N(0, 1)
    stds = [a.std()]
    for layer in range(depth):
        std = weight_std(init_kind, N)
        W = np.random.randn(N, N) * std
        h = W @ a
        a = activate(h, act_kind)
        stds.append(a.std())
    return stds

## Run it for a few activation/init combinations

Naive `N(0,1)` weights blow the signal up almost immediately. Xavier init keeps a `tanh` network's
signal alive, but starves a `relu` network because half the units die at each layer, cutting the
variance roughly in half every layer. He init compensates for exactly that factor of two, and
keeps `relu` healthy across many layers.

In [ ]:
depth = 40

combos = [
    ('naive',  'tanh', 'tab:red'),
    ('xavier', 'tanh', 'tab:blue'),
    ('xavier', 'relu', 'tab:orange'),
    ('he',     'relu', 'tab:green'),
]

curves = [forward_pass(depth, act, init) for init, act, _ in combos]
colors = [c for _, _, c in combos]
labels = [f'{init} init, {act}' for init, act, _ in combos]

plot_variance_curves(curves, colors, labels)
for label, stds in zip(labels, curves):
    print(f'{label:18s}  std: layer 0 = {stds[0]:.3f}  ->  layer {depth} = {stds[-1]:.3e}')